In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# =========================
# 1) Ler a base de dados
# =========================
tabela = pd.read_excel("Banco_de_Fardos.xlsx")
tabela.columns = tabela.columns.str.strip()  # remove espaços extras nos nomes

# Mantém apenas as colunas que você falou
tabela = tabela[["MATERIAL", "NOME_CONCO", "CONVERSAO", "FATOR"]]

# =========================
# 2) Tratar colunas
# =========================
# FATOR precisa ser numérico
tabela["FATOR"] = pd.to_numeric(tabela["FATOR"], errors="coerce").fillna(1.0)

# Criar dicionário para guardar os codificadores
codificadores = {}

# Codificar MATERIAL e NOME_CONCO
for coluna in ["MATERIAL", "NOME_CONCO"]:
    le = LabelEncoder()
    tabela[coluna] = le.fit_transform(tabela[coluna].astype(str))
    codificadores[coluna] = le

# Codificar CONVERSAO (rótulo de classificação)
le_conversao = LabelEncoder()
tabela["CONVERSAO"] = le_conversao.fit_transform(tabela["CONVERSAO"].astype(str))
codificadores["CONVERSAO"] = le_conversao

print("✅ Colunas após tratamento:")
print(tabela.dtypes.head())
print("\nValores únicos de CONVERSAO codificada:", tabela["CONVERSAO"].unique())

# =========================
# 3) Separar features e rótulos
# =========================
X = tabela.drop(columns=["CONVERSAO", "FATOR"])
y1 = tabela["CONVERSAO"]   # classificação
y2 = tabela["FATOR"]       # regressão

# =========================
# 4) Treinar modelos com 100% dos dados
# =========================
modelo_conversao = RandomForestClassifier(random_state=42)
modelo_fator = RandomForestRegressor(random_state=42)

modelo_conversao.fit(X, y1)
modelo_fator.fit(X, y2)

print("\n✅ Treinamento concluído com 100% dos dados!")

# =========================
# 5) Ler novos fardos e aplicar previsão
# =========================
novos_fardos = pd.read_excel("novos_fardos.xlsx")
novos_fardos.columns = novos_fardos.columns.str.strip()
novos_fardos = novos_fardos[["MATERIAL", "NOME_CONCO"]]

# Aplicar mesma codificação
for coluna, le in codificadores.items():
    if coluna in novos_fardos.columns:
        novos_fardos[coluna] = novos_fardos[coluna].map(
            lambda x: le.transform([x])[0] if x in le.classes_ else -1
        )

X_novos = novos_fardos.copy()

# Fazer previsões
prev_conversao = modelo_conversao.predict(X_novos)
prev_fator = modelo_fator.predict(X_novos)

# Montar resultado final
resultado = novos_fardos.copy()
resultado["CONVERSAO"] = le_conversao.inverse_transform(prev_conversao)
resultado["FATOR"] = prev_fator

# Salvar resultado
resultado.to_excel("PlanilhaAtualizada.xlsx", index=False)
print("\n✅ Resultado salvo em PlanilhaAtualizada.xlsx")
